## Evaluating predictions on testing data

In [ ]:
import sys
import importlib as imp
import numpy as np
import torch
import rasterio
import matplotlib.pyplot as plt
from rasterio.windows import Window
from sklearn.metrics import mean_squared_error, mean_absolute_error

from utils import utils
from visualizer import plots
from analyzer import analyze_predictions

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"pytorch version = {torch.__version__}")

In [ ]:
# GET config
EXP_NAME = "exp008"

config = utils.get_config(EXP_NAME)
config["mode"] = "inference"
directory_paths = utils.get_directories(config["machine"])

In [ ]:
nsamples_to_plot = 100_000

# overwrite the inference region
# config["data"]["inference_region"] = [6, 8, 0, 5]

for year in (2020,):
    print(" --- " + str(year) + "---")
    config["inference_years"] = (year,)

    # TILE THE PREDICTIONS TOGETHER
    model_name = utils.get_model_name(config["expname"], config["seed"])
    mosaic_filename = (
        directory_paths["mosaics_dir"]
        + model_name
        + "_"
        + str(config["inference_years"][0])
        + "_mlhfi_mosaic.tif"
    )
    labels_filename = (
        directory_paths["data_dir"]
        + "hii_"
        + str(config["inference_years"][0])
        + "-01-01_uint8.tif"
    )

    with rasterio.open(mosaic_filename) as predict_tif:

        # get bounds of interest for analysis
        lat_s, lat_n, lon_w, lon_e = config["data"]["inference_region"]
        ilat_n, ilon_w = predict_tif.index(lon_w, lat_n)
        ilat_s, ilon_e = predict_tif.index(lon_e, lat_s)
        ilat_n = max(ilat_n, 0)

        print(lat_s, lat_n, lon_w, lon_e)
        window = Window.from_slices((ilat_n, ilat_s + 1), (ilon_w, ilon_e + 1))
        predictions = np.asarray(predict_tif.read(1, window=window), dtype="float")

        with rasterio.open(labels_filename) as label_tif:

            # get correct bounds to evaluate the labels
            lon_w, lat_n = predict_tif.xy(ilat_n, ilon_w, offset="ul")
            lon_e, lat_s = predict_tif.xy(ilat_s, ilon_e, offset="ul")
            ilat_n, ilon_w = label_tif.index(lon_w, lat_n)
            ilat_s, ilon_e = label_tif.index(lon_e, lat_s)

            print(lat_s, lat_n, lon_w, lon_e)
            window = Window.from_slices((ilat_n, ilat_s + 1), (ilon_w, ilon_e + 1))
            labels = np.asarray(label_tif.read(1, window=window), dtype="float")

    # ALIGN and PROCESS PREDICTIONS FOR ANALYAIS
    labels, predictions = analyze_predictions.process_flatten(
        config, labels, predictions, remove_edges=True
    )

    # %%
    # MAKE HISTOGRAMS
    plt.figure(figsize=(20, 4.5))
    plt.subplot(1, 3, 1)
    bins = np.arange(0, 101, 1)
    plt.hist(labels, bins, density=True)
    plt.title(config["expname"] + ": labels")
    plt.ylim(0, 0.05)

    plt.subplot(1, 3, 2)
    bins = np.arange(0, 101, 1)
    plt.hist(predictions, bins, density=True)
    plt.title(config["expname"] + ": predictions")
    plt.ylim(0, 0.05)

    # MAKE SUMMARY FIGURE
    rmse = np.sqrt(mean_squared_error(labels, predictions)).round(4)
    mae = mean_absolute_error(labels, predictions).round(4)
    corr = np.corrcoef(labels, predictions)[0, 1].round(4)

    rng = np.random.default_rng(42)
    iplot = rng.choice(np.arange(len(predictions)), size=nsamples_to_plot)

    plt.subplot(1, 3, 3)
    inc = 2
    plt.hist2d(
        x=labels[iplot],
        y=predictions[iplot],
        density=False,
        norm="log",
        bins=np.arange(0, 100 + inc, inc),
        cmap="Spectral_r",
        vmin=1,
    )
    plt.colorbar()
    plt.plot((0, 100), (0, 100), "-", linewidth=2, color="k", alpha=0.75)
    plt.title(f"{rmse = }, {mae = }, {corr = }")
    plt.xlabel("labels")
    plt.ylabel("predictions")
    plt.xlim(0, 100)
    plt.ylim(0, 100)

    plots.savefig(config, str(year) + "_prediction_metrics")
    plt.show()

In [ ]:
# def get_lats_lons_list(region, tile_len_deg):

#     if isinstance(region, list):

#         lat_s, lat_n, lon_w, lon_e = region
#         lats_list = [np.arange(lat_s + tile_len_deg, lat_n + tile_len_deg, tile_len_deg),]
#         lons_list = [np.arange(lon_w, lon_e, tile_len_deg)]

#     elif isinstance(region, dict):
#         lats_list = region["lats"]
#         lons_list = region["lons"]

#         if len(lats_list) != len(lons_list):

#             assert len(lons_list) == 1 or len(lats_list) == 1

#             if len(lons_list) == 1:
#                 lons_list = np.repeat(lons_list, len(lats_list), axis=0)

#             if len(lats_list) == 1:
#                 lats_list = np.repeat(lats_list, len(lons_list), axis=0)

#     else:
#         raise NotImplementedError

#     return {"lats": lats_list, "lons": lons_list}

# import numpy as np
# d = {
#     "lats": [[0]],
#     "lons": [[0, 10, 30], [90], [33, 44]],
# }

# d = [-40, 40, 0, 20]
# region = get_lats_lons_list(d,10)
# print(region)

In [ ]:
# for lats_list, lons_list in zip(region["lats"], region["lons"]):
#     print(lats_list, lons_list)